# Faster Vehicle Routing: Solving CVRPTW with OCI AI Accelerator Pack

Vehicle Routing Problems (VRPs) are a backbone of real-world logistics: planning efficient delivery routes for a fleet that must serve many customers. They show up everywhere in supply chain operations, such as last‑mile delivery, field service, replenishment, and distribution planning. At scale, even small improvements in routing can translate into meaningful cost and service gains.

This guide provides a **step-by-step** walkthrough for solving a common real-world routing challenge. The **Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)** using the **Vehicle Route Optimizer** from the **OCI AI Accelerator Pack**, powered by **NVIDIA cuOpt** deployed on **Oracle Cloud Infrastructure (OCI)**.

## Table of Contents
1. [Deploy Vehicle Route Optimizer using the OCI AI Accelerator Pack](#1-deploy-vehicle-route-optimizer-using-the-oci-ai-accelerator-pack)
2. [Install Dependencies](#2-install-dependencies)
3. [Set Up Helper Functions](#3-set-up-helper-functions)
4. [Check API Endpoint](#4-check-api-endpoint)
5. [Data Structure](#5-data-structure)  
   - 5.1 [Cost Matrix](#51-cost-matrix)  
   - 5.2 [Fleet Data](#52-fleet-data)  
   - 5.3 [Task Data: Demand and Locations](#53-task-data-demand-and-locations)  
   - 5.4 [Time Windows and Service Times](#54-time-windows-and-service-times)  
6. [Gehring & Homberger Benchmark](#6-gehring--homberger-benchmark)
7. [C1_2_1 Dataset](#7-C1_2_1-Dataset)
   - 7.1 [Download and Visualise the C1_2_1 Dataset](#71-download-and-visualise-the-c1_2_1-dataset)  
   - 7.2 [Initialise cuOpt Payload](#72-initialise-cuopt-payload)  
   - 7.3 [Solve and Visualise (5-Second Time Limit)](#73-solve-and-visualise-5-second-time-limit)  
   - 7.4 [Solve and Visualise (10-Second Time Limit)](#74-solve-and-visualise-10-second-time-limit)  
   - 7.5 [Performance Summary](#75-performance-summary)  
8. [R1_2_1 Dataset](#8-R1_2_1-Dataset)   
   - 8.1 [Download and Visualise the R1_2_1 Dataset](#81-download-and-visualise-the-r1_2_1-dataset)  
   - 8.2 [Initialise cuOpt Payload](#82-initialise-cuopt-payload)  
   - 8.3 [Solve and Visualise (5-Second Time Limit)](#83-solve-and-visualise-5-second-time-limit)  
   - 8.4 [Solve and Visualise (10-Second Time Limit)](#84-solve-and-visualise-10-second-time-limit)  
   - 8.5 [Performance Summary](#85-performance-summary)  
9. [Conclusion](#9-conclusion)
10. [Cleanup](#10-cleanup-recommended)

## 1. Deploy Vehicle Route Optimizer using the OCI AI Accelerator Pack
1. In the **OCI Console**, open the navigation menu and go to:  
   **Analytics & AI** → **AI Accelerator Pack** → **Vehicle Route Optimizer**.
2. Accept the **Oracle Terms of Use** and review/confirm the stack information.  
   ![Stack information](./img/stack-info.png)
3. Complete the required stack variables.
   - In this guide, use the **PoC size**, which deploys a **VM.GPU.A10.2** shape (2× NVIDIA A10 GPUs).
4. To identify Availability Domains (ADs) with A10 capacity:
   - **Governance & Administration** → **Limits, quotas and usage**
   - **Edit Filters**
     - **Service** = Compute
     - **Scope** = Selected AD
   - In **Search**, type **A10**  
   ![Configure variables](./img/configure-variables.png)
5. Click **Deploy**.
   - Deployment typically takes **30–45 minutes**.
   - The process first sets up the **AI Blueprint (OKE)**, then deploys the **AI Accelerator Pack**.
6. After deployment completes:
   - The stack includes **NVIDIA AI Enterprise** and MLOps components such as **Prometheus**, **Grafana**, **MLflow**, and **KEDA**.
   - The **API endpoint** is displayed in the stack **Outputs**.

## 2. Install Dependencies
After deployment completes, set up a Python notebook environment to call the Vehicle Route Optimizer API and visualise results.

- Recommended: **Python 3.10–3.12**
- Ensure network access to the deployed API endpoint (typically via your VCN, bastion, or VPN).

In [ ]:
# The following packages are required to run this notebook:
%pip install -q matplotlib scipy pandas requests

import os
import sys
import requests
import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO)

## 3. Set Up Helper Functions
All utility functions used in this notebook are defined in `cvrptw-utils/utils.py` and imported below.

The module provides the following helpers:

| Function | Purpose |
|----------|---------|
| `solution_eval` | Compare cuOpt results against a best-known solution |
| `plot_routes` | Visualise vehicle routes on a two-panel map |
| `create_from_file` | Parse a Gehring & Homberger benchmark instance file |
| `build_fleet_data` | Build the `fleet_data` section of a cuOpt request payload |
| `build_task_data` | Build the `task_data` section of a cuOpt request payload |
| `build_payload` | Assemble a complete cuOpt request payload |
| `summarise_results` | Print a performance summary table |
| `plot_instance` | Visualise a benchmark dataset (locations + time-window distribution) |
| `solve` | Manage the full request–response cycle for the cuOpt API |

In [ ]:
# Add the cvrptw-utils directory to the Python module search path
sys.path.insert(0, './cvrptw-utils')

from utils import (
    solution_eval,
    plot_routes,
    plot_instance,
    create_from_file,
    build_fleet_data,
    build_task_data,
    build_payload,
    solve,
    summarise_results,
)


## 4. Check API Endpoint

Check that this notebook has access to the Vehicle Route Optimizer API. 

In [ ]:
# Base URL of the cuOpt optimization server
# Update this value to point to your own deployed cuOpt instance 
base_url = "http://<Replace with your cuOpt Endpoint>"

In [ ]:
# Check that the cuOpt server is running and accessible before submitting any requests
health_url = f"{base_url}/cuopt/health"
try:
    response = requests.get(health_url)
    print(f"Health check URL : {health_url}")
    print(f"Status code      : {response.status_code}")
    print(f"Response         : {response.json()}")
except requests.RequestException as e:
    print(f"Health check failed: {e}")

## 5. Data Structure
Before running any optimisation, it’s important to understand the request payload structure. cuOpt expects three primary components:

| Component | Description |
|-----------|-------------|
| **Cost Matrix** | Pairwise travel cost between every pair of locations (e.g., distance, travel time, or a generalized cost). |
| **Fleet Data** | Available vehicles, including start/end depots and capacity constraints (and any other vehicle-level constraints, if applicable). |
| **Task Data** | Customer tasks/stops, including location, demand, time windows, and service time. |

### 5.1 Cost Matrix
In CVRPTW, the solver must know the travel cost between every pair of locations to decide which vehicle visits which customers, and in what order. This pairwise cost information is represented as a **cost matrix** (also called a *distance matrix* or *travel time matrix*).

Specifically, `cost_matrix[i][j]` represents the cost of travelling **from location `i` to location `j`**. The solver uses this matrix to:
- **Evaluate candidate routes**: compute the total travel cost of a proposed sequence of stops.
- **Minimise total cost**: find the assignment of customers to vehicles, and the visit order, that yields the lowest overall travel cost.
- **Respect time windows**: derive arrival times at each stop to verify time window constraints are satisfied.

To illustrate the structure of a cost matrix, we use a simplified **5-location London example** with **1 depot and 4 customers**.

In [ ]:
from scipy.spatial.distance import cdist

names = [
    "King's Cross (Depot)",  # 0 — depot
    "Camden",                # 1
    "Shoreditch",            # 2
    "Elephant & Castle",     # 3
    "Waterloo",              # 4
]
lats    = [51.5308, 51.5390, 51.5226, 51.4958, 51.5031]
lons    = [-0.1238, -0.1426, -0.0782, -0.1000, -0.1145]
demands = [0, 10, 15, 20, 12]           # depot demand is always 0
tws     = [(0, 480), (60, 120), (90, 180), (30, 150), (120, 240)]
n       = len(names)                    # 5 locations (1 depot + 4 customers)

london_df = pd.DataFrame([
    {
        'vertex'        : i,
        'xcord'         : lons[i],   
        'ycord'         : lats[i],   
        'demand'        : demands[i],
        'earliest_time' : tws[i][0],
        'latest_time'   : tws[i][1],
        'service_time'  : 5,         
    }
    for i in range(n)
])

# Compute the Euclidean distance matrix from (lat, lon) coordinates.
coords = np.column_stack([lats, lons])
dist_matrix_example = cdist(coords, coords, metric='euclidean')

# Display the full 5×5 distance matrix with location labels
short_names = ['Depot', 'Camden', 'Shoreditch', 'Elephant', 'Waterloo']
dist_df = pd.DataFrame(
    dist_matrix_example.round(4),
    index=short_names,
    columns=short_names,
)
print('5×5 Euclidean distance matrix (degrees):')
print(dist_df.to_string())
print()

In a production application, Euclidean distance is only a rough approximation because it ignores road topology, one-way streets, speed limits, and real driving paths. For more accurate routing, compute **road network distances / travel times** using a routing engine or API.

### 5.2 Fleet Data
Fleet data describes the **vehicles** available to serve customers. Each vehicle is characterised by:
- **Start and end location**: where the vehicle departs from and returns to (both set to depot = location 0)
- **Capacity**: the maximum total demand the vehicle can carry on a single route

The `capacities` field is a list of lists to support multi-dimensional capacity constraints (e.g., weight + volume), though here we use a single dimension.

In [ ]:
# Build fleet data for the London example: 2 vehicles, capacity 50 each
fleet_data_example = build_fleet_data(n_vehicles=2, vehicle_capacity=50)
print('fleet_data keys:', list(fleet_data_example.keys()))
print('Vehicle start/end locations (depot=0):', fleet_data_example['vehicle_locations'])
print('Vehicle capacities:', fleet_data_example['capacities'])

### 5.3 Task Data: Demand and Locations
Task data describes the **delivery tasks** (customer orders) that vehicles must complete. Each task is associated with:
- A **location index** that maps to a row/column in the cost matrix
- A **demand** value representing how much cargo must be delivered

In [ ]:
# Build task data for the London example
n_locations_london = n - 1  # exclude depot
task_data_example = build_task_data(london_df, n_locations_london)

print('task_data keys:', list(task_data_example.keys()))
print('Customer location indices:', task_data_example['task_locations'])
print('Customer demands:', task_data_example['demand'])


### 5.4 Time Windows and Service Times
Time windows are one of the defining constraints of CVRPTW. They ensure that each customer is visited within an acceptable time range:
- **`earliest_time`** (`e_i`): earliest time a vehicle may begin servicing customer `i`. If the vehicle arrives before this time, it must **wait** at the location.
- **`latest_time`** (`l_i`): latest time a vehicle may **start** servicing customer `i`. Arriving after this time violates the constraint.
- **`service_time`** (`s_i`): time required to complete service at location `i` (e.g., unloading goods). The vehicle departs at `start_service_time + service_time`.

These values determine the **feasibility** of any route, vehicles must reach each customer within its time window, given travel times derived from the cost matrix.

In [ ]:
print('Time windows and service times (customers):')
print(f'{"Customer":<25} {"Earliest":>10} {"Latest":>10} {"Service":>10}')
print('-' * 57)
for idx, tw in enumerate(task_data_example['task_time_windows'], start=1):
    svc = task_data_example['service_times'][idx - 1]
    print(f'C{idx}: {names[idx]:<22} {tw[0]:>10} {tw[1]:>10} {svc:>10}')

## 6. Gehring & Homberger Benchmark
To ground the discussion, we use two standard **Gehring & Homberger VRPTW** benchmark instances with **200 customers** each.

We follow the common benchmark objective: **(1) minimise fleet size**, then **(2) minimise total travel cost**.

The instances are provided in the standard Gehring & Homberger text format and can be downloaded from the **SINTEF TOP benchmark repository** (a widely used source for VRPTW benchmark instances).

### 6.1 File Format Overview
- **Line 5**: number of vehicles and vehicle capacity
- **Lines 10+**: one row per location (depot + 200 customers)

| Column | Description |
|--------|-------------|
| `vertex` | Location index (`0` = depot) |
| `xcord`, `ycord` | Benchmark coordinates (abstract grid units) |
| `demand` | Units of goods to deliver |
| `earliest_time` | Earliest service start time |
| `latest_time` | Latest service start time |
| `service_time` | Time spent servicing the customer |

### 6.2 Solver Time Limits
We run each instance with two different solver time limits to show how solution quality changes with more compute time:
- **5-second limit**: suitable for time-sensitive scenarios (e.g., near-real-time re-optimisation)
- **10-second limit**: moderate budget that allows additional route refinement

The `solver_config` field controls the solver time budget.

### 6.3 Metrics Tracked
For each run, we track:
- **Vehicles used**: fewer vehicles generally reduces fleet operating costs (drivers, fuel, maintenance).
- **Total cost**: sum of route travel costs (a proxy for distance / driving time in these benchmarks).

### 6.4 Instance Characteristics
**C1 class**:
- Clustered customer locations (geographic clusters often make routing more efficient).
- Tight time windows (narrow service windows increase scheduling complexity).
- Homogeneous fleet (identical capacities, common depot).

**R1 class**:
- Random customer locations (uniform scattering makes efficient grouping harder).
- Tight time windows.
- Higher travel cost (typically more inter-stop travel than clustered instances).

### 6.5 Instances Used
| Instance | Distribution | Customers | Best-known vehicles | Best-known cost |
|----------|--------------|-----------|---------------------|-----------------|
| `C1_2_1` | Clustered    | 200       | 20                  | 2704.57         |
| `R1_2_1` | Random       | 200       | 20                  | 4784.11         |

Best known reference results are published by [SINTEF TOP](https://www.sintef.no/projectweb/top/vrptw/200-customers/).

## 7. C1_2_1 Dataset
### 7.1 Download and Visualise the C1_2_1 Dataset

In [ ]:
# ── Download the Gehring & Homberger 200-customer benchmark data ─────
!mkdir -p ./cvrptw_data
!wget -q https://www.sintef.no/globalassets/project/top/vrptw/homberger/200/homberger_200_customer_instances.zip -O ./cvrptw_data/homberger_data.zip
!unzip -q -o ./cvrptw_data/homberger_data.zip -d ./cvrptw_data/

# C1_2_1: clustered instance, 200 customers, instance 1
homberger_c1_file = "./cvrptw_data/C1_2_1.TXT"
if not os.path.exists(homberger_c1_file):
    raise FileNotFoundError(
        f"Could not find {homberger_c1_file}. "
        "Please check that the download succeeded."
    )

best_known_solution_c1 = {
    "n_vehicles": 20,  
    "cost": 2704.57      
}

orders_c1, vehicle_capacity_c1, n_vehicles_c1 = create_from_file(homberger_c1_file)
n_locations_c1 = orders_c1['demand'].shape[0] - 1

print('Number of customers          :', n_locations_c1)
print('Number of vehicles available :', n_vehicles_c1)
print('Capacity of each vehicle     :', vehicle_capacity_c1)
print('Best-known vehicles          :', best_known_solution_c1['n_vehicles'])
print('Best-known cost              :', best_known_solution_c1['cost'])
print('\nFirst 5 rows of the C1_2_1 DataFrame:')
print(orders_c1.head(5).to_string(index=False))


In [ ]:
# Plot the downloaded data
plot_instance(orders_c1, 'C1_2_1', 'Clustered')

The first benchmark is `C1_2_1`, where **C** denotes *clustered* customer locations. Customers form distinct geographic clusters rather than being spread uniformly across the service area. The plots show the spatial clustering and the time-window constraints that make CVRPTW challenging.

### 7.2 Initialise cuOpt Payload

With the benchmark data loaded, we now build the cuOpt **payload** using the `build_payload` helper from `cvrptw-utils/utils.py`. This single call assembles the **cost matrix**, **fleet data**, and **task data**(including time windows and service times) in one step.

In [ ]:
payload_c1 = build_payload(
    orders_c1, n_locations_c1, n_vehicles_c1, vehicle_capacity_c1
)

print('Payload keys:', list(payload_c1.keys()))
print(f'Cost matrix size: {len(payload_c1["cost_matrix_data"]["data"]["0"])} x '
      f'{len(payload_c1["cost_matrix_data"]["data"]["0"][0])}')
print('Fleet vehicles  :', len(payload_c1['fleet_data']['vehicle_locations']))
print('Task locations  :', len(payload_c1['task_data']['task_locations']))


### 7.3 Solve and Visualise (5-Second Time Limit)
This run gives the solver only **5 seconds** to find a solution.

In [ ]:
solve_results_c1 = {}

solution_data_c1 = solve(base_url, payload_c1, 5)
if solution_data_c1.get('status') == 0:
    n_veh_used = solution_data_c1.get('num_vehicles')
    total_cost = solution_data_c1.get('solution_cost')

    solve_results_c1[5] = {
        'num_vehicles' : n_veh_used,
        'solution_cost': total_cost,
    }

    solution_eval(n_veh_used, total_cost, best_known_solution_c1)
    plot_routes(
        solution_data_c1,
        orders_c1,
        title='C1_2_1 Clustered Benchmark (5 s)',
        max_routes=20,
        detail_routes=20
    )
else:
    print(f"Optimisation failed or returned status: {solution_data_c1.get('status')}")

### 7.4 Solve and Visualise (10-Second Time Limit)
This run gives the solver **10 seconds**, allowing more time to refine routes compared to the 5-second run.

In [ ]:
solution_data_c1 = solve(base_url, payload_c1, 10)
if solution_data_c1.get('status') == 0:
    n_veh_used = solution_data_c1.get('num_vehicles')
    total_cost = solution_data_c1.get('solution_cost')

    solve_results_c1[10] = {
        'num_vehicles' : n_veh_used,
        'solution_cost': total_cost,
    }

    solution_eval(n_veh_used, total_cost, best_known_solution_c1)
    plot_routes(
        solution_data_c1,
        orders_c1,
        title='C1_2_1 Clustered Benchmark (10 s)',
        max_routes=20,
        detail_routes=20
    )
else:
    print(f"Optimisation failed or returned status: {solution_data_c1.get('status')}")

### 7.5 Performance Summary
The table below compares cuOpt's solution quality across the two time limits (2 s, 10 s) against the published best-known solution for the `C1_2_1` benchmark.

In [ ]:
summary_c1 = summarise_results(
    solve_results_c1,
    best_known_solution_c1,
    benchmark_name='200-Customer C1_2_1 Benchmark (Clustered)'
)

Best known solutions can typically take hours or even days to compute, and they represent the strongest results reported by academic researchers across a range of algorithms and significant computational resources. Achieving the best-known result in seconds, given that the sheer combination of possible routes is astronomically high, demonstrates the power to find high quality routes that satisfy all constraints within a tight runtime budget. 

## 8. R1_2_1 Dataset
### 8.1 Download and Visualise the R1_2_1 Dataset

In [ ]:
# R1_2_1: randomly distributed instance, 200 customers, instance 1
homberger_r1_file = "./cvrptw_data/R1_2_1.TXT"
if not os.path.exists(homberger_r1_file):
    raise FileNotFoundError(
        f"Could not find {homberger_r1_file}. "
        "Please check that the benchmark zip was downloaded and unzipped."
    )

best_known_solution_r1 = {
    "n_vehicles": 20,
    "cost": 4784.11,
}

orders_r1, vehicle_capacity_r1, n_vehicles_r1 = create_from_file(homberger_r1_file)
n_locations_r1 = orders_r1['demand'].shape[0] - 1

print('Number of customers          :', n_locations_r1)
print('Number of vehicles available :', n_vehicles_r1)
print('Capacity of each vehicle     :', vehicle_capacity_r1)
print('\nFirst 5 rows of the R1_2_1 DataFrame:')
print(orders_r1.head(5).to_string(index=False))

In [ ]:
# Plot the downloaded data
plot_instance(orders_r1, 'R1_2_1', 'Random')

`R1_2_1`, where **R** denotes *random* customer locations, is typically harder: customers are spread more uniformly, increasing inter-stop travel and making tight time windows more difficult to satisfy without detours or waiting.

### 8.2 Initialise cuOpt Payload
Build the cuOpt payload (cost matrix, fleet data, task data) for the benchmark instance.

In [ ]:
payload_r1 = build_payload(orders_r1, n_locations_r1, n_vehicles_r1, vehicle_capacity_r1)
print('Payload keys:', list(payload_r1.keys()))
print(f'Cost matrix size: {len(payload_r1["cost_matrix_data"]["data"]["0"])} x '
      f'{len(payload_r1["cost_matrix_data"]["data"]["0"][0])}')
print('Fleet vehicles  :', len(payload_r1['fleet_data']['vehicle_locations']))
print('Task locations  :', len(payload_r1['task_data']['task_locations']))

### 8.3 Solve and Visualise (5-Second Time Limit)

In [ ]:
solve_results_r1 = {}

solution_data_r1 = solve(base_url, payload_r1, 5)
if solution_data_r1.get('status') == 0:
    n_veh_used = solution_data_r1.get('num_vehicles')
    total_cost = solution_data_r1.get('solution_cost')

    solve_results_r1[5] = {
        'num_vehicles' : n_veh_used,
        'solution_cost': total_cost,
    }

    solution_eval(n_veh_used, total_cost, best_known_solution_r1)
    plot_routes(
        solution_data_r1,
        orders_r1,
        title='R1_2_1 Random Benchmark (5 s)',
        max_routes=20,
        detail_routes=20
    )
else:
    print(f"Optimisation failed or returned status: {solution_data_r1.get('status')}")

### 8.4 Solve and Visualise (10-Second Time Limit)

In [ ]:
solution_data_r1 = solve(base_url, payload_r1, 10)
if solution_data_r1.get('status') == 0:
    n_veh_used = solution_data_r1.get('num_vehicles')
    total_cost = solution_data_r1.get('solution_cost')

    solve_results_r1[10] = {
        'num_vehicles' : n_veh_used,
        'solution_cost': total_cost,
    }

    solution_eval(n_veh_used, total_cost, best_known_solution_r1)
    plot_routes(
        solution_data_r1,
        orders_r1,
        title='R1_2_1 Random Benchmark (10 s)',
        max_routes=20,
        detail_routes=20
    )
else:
    print(f"Optimisation failed or returned status: {solution_data_r1.get('status')}")

### 8.5 Performance Summary
Compare cuOpt solution quality across the time limits against the best-known solution.

In [ ]:
summary_r1 = summarise_results(
    solve_results_r1,
    best_known_solution_r1,
    benchmark_name='200-Customer R1_2_1 Benchmark (Random)'
)

Note that this dataset is more interesting from an academic perspective, as customers are likely to be clustered in residential areas rather than randomly scattered over the city.

This run should have reached a high‑quality answer in just 5 seconds by matching the best-known fleet size of 20 vehicles. This is often the primary requirement because fleet size drives the largest fixed overhead (drivers, shifts, and dispatch capacity). Extending the time limit to 10 seconds improves route efficiency, but the incremental savings typically have less business impact than getting the vehicle count right.

In a real-world business setting, chasing a mathematically proven optimum often leads to diminishing returns. While academic benchmarks focus on finding the single most efficient solution regardless of computation time, in practice it is usually more realistic to establish a baseline based on existing business processes and then measure improvements in fleet size, SLA adherence, and travel cost under tight time limits.

That said, you have full control over this trade-off by adjusting the solve time to balance speed and incremental optimisation. For example, in a dynamic routing scenario with real-time traffic updates, you may prioritise faster runtimes to quickly adapt routes, even if it means accepting a slightly higher cost. Conversely, for static planning where routes are set well in advance, you might allow for longer runtimes to achieve the lowest possible cost. The flexibility of the solution allows you to adjust this trade-off based on your operational needs.

## 9. Conclusion

We showed how to solve the Capacitated Vehicle Routing Problem with Time Windows (CVRPTW) using NVIDIA cuOpt deployed with the OCI AI Accelerator Pack. We started from a practical enterprise routing scenario and then validated solution quality on two Gehring and Homberger benchmark instances, C1_2_1 (clustered) and R1_2_1 (random).

Across both benchmarks, cuOpt delivered near optimal results in seconds under tight solver time budgets. On the clustered C1_2_1 instance, it matched the published best-known solution in under 10 seconds, reaching the best-known fleet size and cost. On the more challenging random R1_2_1 instance, it matched the best-known fleet size within 5 seconds and reduced the cost gap further with a 10 second solve. These results demonstrate that high quality routing plans can be generated quickly enough to support operational decision making, not just offline optimisation.

The OCI AI Accelerator Pack provides a production-oriented foundation for running GPU accelerated optimisation on Oracle Cloud Infrastructure, enabling teams to move from proof of concept to deployment with consistent architecture patterns. Whether you are optimising last mile delivery routes, field service schedules, or other logistics workflows, OCI offers powerful tools that can take Kaizen to the next level. Happy optimising!


## 10. Cleanup
To avoid ongoing GPU/OKE costs after testing:
1. In the OCI Console, locate the Resource Manager stack used to deploy the Vehicle Route Optimizer.
2. Run **Destroy** on the stack (or follow your team’s standard teardown process).
3. Verify that GPU compute resources, load balancers, block volumes, and any related networking components were deleted as expected.